## Packages Import


In [1]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

## Apollo Scraper

In [2]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [ ]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)


In [ ]:
print(response.status_code)
print(response.text[:500])

In [5]:
page_dom = BeautifulSoup(response.text, 'html.parser')


In [ ]:
group = page_dom.select_one("div.grupa")
if group:
    group = group.get_text(strip=True)
else:
    group = "unknown"
print(group)

### TABLE DATA FRAME

In [7]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

### FILTER

In [8]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

### SPLIT DAY TIME

In [9]:
classes[['Day','Start time', 'hyphen', 'End time', "Duration"]] = classes['Dzień, godzina'].str.split(' ',expand=True)
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

### CLEAN SALA

In [10]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).+",
    r"\1",
    regex=True
)

###  EXPORT

In [ ]:
if not os.path.exists("schedules"):
    os.makedirs("schedules")
classes.to_csv(f"schedules/{group}.csv")

classes